# Admin Monitoring Notebook

This notebook demonstrates how you, as an Administrator, can use Snowflake notebooks. Over time, you may have collected or will collect useful queries/code snippets that can be run in worksheets. Notebooks are a nifty way to group scripts in one place, run them, see the results immediately, and/or visualize the results using tools such as matplotlib. For demonstration purposes, we've included a couple of charts and some sample Python code. You can even schedule notebooks to run at a specific time.

📚 Learn:

❄️ Open a pre-built Snowflake Notebook with sample monitoring code

❄️ Run the code in a notebook cell to see the query results

❄️ Learn how to query the snowflake.account_usage.query_history view and report on it

❄️ Schedule the notebook to run at a particular time

📌 Note:
Each time you run the notebook, remember to run the setup context cell below. If the notebook is idle, the session may time out; simply re-run the setup cell to restore your context.

## 1.0 Get Started

### 1.1 Setup context.
Run the following cells to set up the context for your notebook. This must be done each time you work on this notebook.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_DB'

In [ ]:
%%sql -r setup_context
USE ROLE SYSADMIN;
CREATE DATABASE IF NOT EXISTS {{user}}_adm_db
COMMENT='Database for Admin course labs';

CREATE WAREHOUSE IF NOT EXISTS {{user}}_adm_wh
    WAREHOUSE_SIZE=XSmall
    INITIALLY_SUSPENDED=True
    AUTO_SUSPEND=300
    COMMENT='Warehouse for Admin course labs';

USE ROLE SYSADMIN;
USE WAREHOUSE {{user}}_adm_wh;
CREATE OR REPLACE SCHEMA {{user}}_adm_db.monitoring;
USE SCHEMA {{user}}_adm_db.monitoring;

### 1.2 Check context.
Run the following commands to validate that the context for your notebook was set correctly.

In [ ]:
print('Your current CONTEXT information:')
print('---------------------------------')
print(session)
print('Your current USER is ' + user)

### 1.3 Set query tag.

Set up a query tag for your session and confirm it is set correctly.

In [ ]:
%%sql -r setup_query_sql
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: Monitoring';
SHOW PARAMETERS LIKE 'query_tag' IN SESSION;

## 2.0 Security
### 2.1 Authentication methods.
Run the following command to get a breakdown of the authentication methods used in your Snowflake account.

In [ ]:
%%sql -r authentication_methods_sql
SELECT
   first_authentication_factor || ' ' || nvl(second_authentication_factor, '') AS authentication_method
   , count(*)
FROM snowflake.account_usage.login_history
WHERE is_success = 'YES'
AND user_name != 'WORKSHEETS_APP_USER'
GROUP BY authentication_method
ORDER BY count(*) DESC;

### 2.2 Login failures.

Run the next query to find out which login failures are a common occurrence. We have excluded some usernames and error messages to simplify the output as they are internal to Snowflake. This cell is referenced later as `login_failures_sql` to visualize the data using a Python cell and matplotlib to create a bar chart.

In [ ]:
%%sql -r login_failures_sql
SELECT
   ERROR_MESSAGE, count(*)
FROM snowflake.account_usage.login_history
WHERE is_success = 'NO'
AND user_name != 'WORKSHEETS_APP_USER' AND ERROR_MESSAGE NOT LIKE 'OVERFLOW%'
GROUP BY ERROR_MESSAGE;

### 2.3 Chart the results of the last query by referencing its dataset.
The following Python cell uses the matplotlib library to create a visual representation of the login failures as a bar chart. It reads the `login_failures_sql` SQL cell result, which is already a pandas DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

chart_data = login_failures_sql.to_pandas() if not isinstance(login_failures_sql, pd.DataFrame) else login_failures_sql

plt.figure(figsize=(10, 5))
plt.barh(chart_data['ERROR_MESSAGE'], chart_data['COUNT(*)'], color='#FF6B6B')
plt.xlabel('Count')
plt.ylabel('Error Message')
plt.title('Failed Login Attempts by Error Type')
plt.tight_layout()
plt.show()

### 2.4 Most recent login attempts without MFA.
The following query shows us how many login attempts have been made without MFA. Ideally this should produce no results.

In [ ]:
%%sql -r login_attempts
SELECT
  TO_CHAR(event_timestamp, 'YYYY-MM-DD') AS date_time, user_name
FROM
  snowflake.account_usage.login_history
WHERE
  first_authentication_factor = 'PASSWORD'
  AND second_authentication_factor IS NULL
ORDER BY
  date_time;

### 2.5 Create a copy of the snowflake.account_usage.login_history view.

The snowflake.account_usage.login_history view holds the login history for the last 365 days. If you need to retain it longer for compliance reasons, you may want to create a backup and capture any changes. There are several ways to meet this requirement. In the next few cells, we demonstrate how you can create a backup table for login_history, append any new records to the backup table, and then schedule the notebook to run once a day. As an administrator you will want to make sure this audit table is well protected from unauthorized access.

In [ ]:
%%sql -r login_history_create
CREATE DATABASE IF NOT EXISTS {{user}}_ADM_DB;
CREATE SCHEMA IF NOT EXISTS {{user}}_ADM_DB.MONITORING;
CREATE TABLE IF NOT EXISTS {{user}}_ADM_DB.MONITORING.LOGIN_HISTORY_ARCHIVE
AS SELECT *
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY;

### 2.6 Append new records to the archive table.
Run the following command to append the new records.

In [ ]:
%%sql -r insrt_command_sql
INSERT INTO {{user}}_ADM_DB.MONITORING.LOGIN_HISTORY_ARCHIVE
SELECT *
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY
WHERE EVENT_TIMESTAMP >= (SELECT MAX(EVENT_TIMESTAMP) FROM {{user}}_ADM_DB.MONITORING.LOGIN_HISTORY_ARCHIVE);

### 2.7 Report on total logins for each day.

Write a query to capture the total logins for each day.

In [ ]:
%%sql -r logins_by_date
SELECT TO_CHAR(EVENT_TIMESTAMP,'DD-MON-YY') LOGINS_BY_DATE, COUNT(USER_NAME) total_logins
FROM {{user}}_ADM_DB.MONITORING.LOGIN_HISTORY_ARCHIVE
GROUP BY LOGINS_BY_DATE;

### 2.8 Create a line chart to demo the logins per day.

Use matplotlib to build a chart to visualize the logins by day. It reads the `logins_by_date` SQL cell result.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

chart_data = logins_by_date.to_pandas() if not isinstance(logins_by_date, pd.DataFrame) else logins_by_date

plt.figure(figsize=(10, 5))
plt.plot(chart_data['LOGINS_BY_DATE'], chart_data['TOTAL_LOGINS'], color='#33C4FF')
plt.xlabel('Date')
plt.ylabel('Total Logins')
plt.title('Logins by Date')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 2.9 Schedule notebook to run daily.

There may be instances where you have a series of commands that you want to schedule to run at a particular interval. In Workspace notebooks, you schedule a notebook either through Snowsight, or by deploying it as a notebook project and creating a task.

Use the UI element **create schedule** to schedule the task:

* In this notebook, click on the calendar icon next to the Share button in the upper right
* Example the schedule details:
    * **Name** - {{login}}_admin_monitoring_schedule e.g. raven_admin_monitoring_schedule
    * **Owner role** - adm_role
    * **Location** - {{login}}_adm_db.monitoring
    * **Frequency** - daily
    * **At** - 7:00am
    * **Query Warehouse** - {{login}}_adm_wh
    * **External access integrations** - ALLOW_ALL_EAI (Does not apply)
* Click Create
* Click on the calendar icon again
    * Notice how a task has been created for your schedule
* Click the **Deploy changes** button to add this task to your schedule

* Tasks will be covered in a later module
    * Alternately, you could deploy this notebook as a Notebook Project, then create a task that runs it on a schedule with `EXECUTE NOTEBOOK PROJECT`.

### 2.10 Shut down the notebook.
When you are finished, disconnect the notebook session from the connection menu in the top left corner.

## Key Takeaways

❄️ Notebooks are a versatile development environment.

❄️ Query results can be visualized using Python tools such as matplotlib.

❄️ You can persist data from snowflake.account_usage views if necessary.

❄️ Notebooks can be scheduled to run at a particular time of the day.